## Instruction/response data generation for SFT

In [1]:
from sft_data_generation import generate_training_example
from datasets import load_dataset

ds = load_dataset("wanglab/cafa5", name="cafa5_reasoning", cache_dir="cafa5_reasoning_cache")
train_df = ds['train'].to_pandas()

example_prompt_template = {
        'system_prompt': "System: You are a scientific assistant specialized in protein function predictions. Given a protein's sequence embeddings and the organism it is associated with, predict the Gene Ontology (GO) terms describing its molecular functions (MF), biological processes this protein is involved in (BP), and cellular components where this protein is found (CC). Do this in a step-by-step manner, by traversing the GO graph from the root term of each subontology (depth 0) all the way to the leaf terms. At each depth, predict the child terms of each GO term that are associated with this protein. Once correct children GO terms are identified, repeat the same process for these GO terms at the next depth of the GO graph. After traversing the GO graph in each requested subontology, list all predicted GO terms, ordered by depth starting from the root. Finally, give a short summary of the protein's function based on the predicted GO terms.",
        
        'user_prompt': "User: Predict {aspect_str} GO terms for this protein:\nprotein sequence embeddings: <embeddings>\norganism: {organism}"
    }

go-basic.obo: fmt(1.2) rel(2023-01-01) 46,739 Terms; optional_attrs(def relationship)


tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
prot_id = 'A0A078CGE6'
protein_data = train_df[train_df['protein_id'] == prot_id].iloc[0]

training_example = generate_training_example(protein_data, example_prompt_template)
print(training_example)

System: You are a scientific assistant specialized in protein function predictions. Given a protein's sequence embeddings and the organism it is associated with, predict the Gene Ontology (GO) terms describing its molecular functions (MF), biological processes this protein is involved in (BP), and cellular components where this protein is found (CC). Do this in a step-by-step manner, by traversing the GO graph from the root term of each subontology (depth 0) all the way to the leaf terms. At each depth, predict the child terms of each GO term that are associated with this protein. Once correct children GO terms are identified, repeat the same process for these GO terms at the next depth of the GO graph. After traversing the GO graph in each requested subontology, list all predicted GO terms, ordered by depth starting from the root. Finally, give a short summary of the protein's function based on the predicted GO terms.

User: Predict Molecular Function, Cellular Component, and Biologic

In [3]:
prot_id = 'A0A0B4K7H5'
protein_data = train_df[train_df['protein_id'] == prot_id].iloc[0]

training_example = generate_training_example(protein_data, example_prompt_template)
print(training_example)

System: You are a scientific assistant specialized in protein function predictions. Given a protein's sequence embeddings and the organism it is associated with, predict the Gene Ontology (GO) terms describing its molecular functions (MF), biological processes this protein is involved in (BP), and cellular components where this protein is found (CC). Do this in a step-by-step manner, by traversing the GO graph from the root term of each subontology (depth 0) all the way to the leaf terms. At each depth, predict the child terms of each GO term that are associated with this protein. Once correct children GO terms are identified, repeat the same process for these GO terms at the next depth of the GO graph. After traversing the GO graph in each requested subontology, list all predicted GO terms, ordered by depth starting from the root. Finally, give a short summary of the protein's function based on the predicted GO terms.

User: Predict Cellular Component and Biological Process GO terms f

In [4]:
prot_id = 'Q9Y7G6'
protein_data = train_df[train_df['protein_id'] == prot_id].iloc[0]

training_example = generate_training_example(protein_data, example_prompt_template)
print(training_example)

System: You are a scientific assistant specialized in protein function predictions. Given a protein's sequence embeddings and the organism it is associated with, predict the Gene Ontology (GO) terms describing its molecular functions (MF), biological processes this protein is involved in (BP), and cellular components where this protein is found (CC). Do this in a step-by-step manner, by traversing the GO graph from the root term of each subontology (depth 0) all the way to the leaf terms. At each depth, predict the child terms of each GO term that are associated with this protein. Once correct children GO terms are identified, repeat the same process for these GO terms at the next depth of the GO graph. After traversing the GO graph in each requested subontology, list all predicted GO terms, ordered by depth starting from the root. Finally, give a short summary of the protein's function based on the predicted GO terms.

User: Predict Molecular Function GO terms for this protein:
protei

### GO graph arranged by depth

In [6]:
import graphviz
from collections import defaultdict
from sft_data_generation import get_term_depth, godag, go2reldepth

def plot_go_with_relationships(go_ids, godag, go2reldepth, output_file="go_custom_plot", dpi=300):
    dot = graphviz.Digraph(comment='GO Terms', format='png')
    dot.attr(ranksep='0.5')  # Increase vertical spacing between ranks
    dot.attr(nodesep='1')  # Horizontal spacing between nodes
    # dot.attr(splines='ortho')  # Use orthogonal edges for cleaner look (optional)
    dot.attr(rankdir='BT')  # Bottom to top
    dot.attr(dpi=str(dpi))  # Set DPI
    
    # Function to create safe node ID
    def safe_id(go_id):
        return go_id.replace(':', '_')
    
    # Group nodes by depth
    nodes_by_depth = defaultdict(list)
    for go_id in go_ids:
        if go_id in godag:
            depth = get_term_depth(go_id, go2reldepth)  # Pass go2reldepth
            nodes_by_depth[depth].append(go_id)
    
    # Add nodes grouped by depth
    for depth in sorted(nodes_by_depth.keys()):
        # Create a subgraph for each depth level with same rank
        with dot.subgraph() as s:
            s.attr(rank='same')  # This is the key - puts all nodes at same visual level
            
            for go_id in nodes_by_depth[depth]:
                go_term = godag[go_id]
                level = go_term.level if hasattr(go_term, 'level') else 'N/A'
                
                # Create multi-line label with depth and level info
                label = f"{go_id}\n{go_term.name}\nDepth: {depth} | Level: {level}"
                
                # Color nodes by depth for better visualization
                if depth == 0:
                    fillcolor = 'lightgreen'
                else:
                    fillcolor = 'lightblue'
                
                s.node(safe_id(go_id), label, shape='box', style='rounded,filled',
                      fillcolor=fillcolor, fontsize='10')
    
    # Add edges
    for go_id in go_ids:
        if go_id in godag:
            go_term = godag[go_id]
            
            # Add is_a relationships (solid lines)
            for parent in go_term.parents:
                if parent.id in go_ids:
                    dot.edge(safe_id(go_id), safe_id(parent.id), color='black', penwidth='1.5')
            
            # Add part_of relationships (dashed lines)
            if hasattr(go_term, 'relationship') and 'part_of' in go_term.relationship:
                for parent in go_term.relationship['part_of']:
                    if parent.id in go_ids:
                        dot.edge(safe_id(go_id), safe_id(parent.id), 
                                color='blue', style='solid', label='part_of', penwidth='1.5')
    
    # Render with high DPI
    dot.render(output_file, cleanup=True)
    print(f"Plot saved as {output_file}.png with {dpi} DPI")

# Call it with go2reldepth parameter
selected_go_ids = train_df[train_df['protein_id'] == prot_id]['go_mf'].iloc[0]
plot_go_with_relationships(selected_go_ids, godag, go2reldepth, output_file="go_by-depth", dpi=300)

Plot saved as go_by-depth.png with 300 DPI


### Go graph arranged by level

In [8]:
from sft_data_generation import godag, go2reldepth, ASPECT_ROOT_TERMS

def plot_go_with_relationships(go_ids, godag, go2reldepth, output_file="go_custom_plot", dpi=300):
    dot = graphviz.Digraph(comment='GO Terms', format='png', engine='dot')  # Explicitly use dot engine
    dot.attr(rankdir='TB')  # Changed from BT to TB - Top to Bottom
    dot.attr(ranksep='1.0')  # Increase vertical spacing
    dot.attr(nodesep='0.5')  # Horizontal spacing
    dot.attr(dpi=str(dpi))
    # Force strict ranking
    dot.attr(newrank='true')  # This is the key attribute!
    
    # Function to create safe node ID
    def safe_id(go_id):
        return go_id.replace(':', '_')
    
    # Group nodes by level
    nodes_by_level = defaultdict(list)
    for go_id in go_ids:
        if go_id in godag:
            level = godag[go_id].level
            nodes_by_level[level].append(go_id)
    
    # Add invisible nodes to enforce ranking if needed
    max_level = max(nodes_by_level.keys())
    
    # Add nodes grouped by level with strict ranking
    for level in sorted(nodes_by_level.keys()):
        # Create a subgraph for each level
        with dot.subgraph(name=f'level_{level}') as s:
            s.attr(rank='same')
            
            # Add an invisible node to anchor this level
            s.node(f'anchor_{level}', '', shape='none', width='0', height='0')
            
            for go_id in nodes_by_level[level]:
                go_term = godag[go_id]
                depth = get_term_depth(go_id, go2reldepth)
                
                # Clean the term name - remove underscores if it's a root term
                term_name = go_term.name
                if go_id in ASPECT_ROOT_TERMS.values():
                    term_name = term_name.replace('_', ' ')
                
                # Create multi-line label
                label = f"{go_id}\\n{term_name}\\nDepth: {depth} | Level: {level}"
                
                # Color nodes - root is green, everything else is light blue
                if level == 0:
                    fillcolor = 'lightgreen'
                else:
                    fillcolor = 'lightblue'
                
                s.node(safe_id(go_id), label, shape='box', style='rounded,filled',
                      fillcolor=fillcolor, fontsize='10')
    
    # Add invisible edges between anchor nodes to enforce level ordering
    sorted_levels = sorted(nodes_by_level.keys())
    for i in range(len(sorted_levels) - 1):
        dot.edge(f'anchor_{sorted_levels[i]}', f'anchor_{sorted_levels[i+1]}', 
                style='invis', weight='100')
    
    # Add edges
    for go_id in go_ids:
        if go_id in godag:
            go_term = godag[go_id]
            
            # Add is_a relationships (solid lines)
            for parent in go_term.parents:
                if parent.id in go_ids:
                    dot.edge(safe_id(go_id), safe_id(parent.id), 
                            color='black', penwidth='1.5', weight='1')
            
            # Add part_of relationships (dashed lines)
            if hasattr(go_term, 'relationship') and 'part_of' in go_term.relationship:
                for parent in go_term.relationship['part_of']:
                    if parent.id in go_ids:
                        dot.edge(safe_id(go_id), safe_id(parent.id), 
                                color='blue', style='solid', label='part_of', 
                                penwidth='1.5', weight='1')
    
    # Render with high DPI
    dot.render(output_file, cleanup=True)
    print(f"Plot saved as {output_file}.png with {dpi} DPI")

selected_go_ids = train_df[train_df['protein_id'] == prot_id]['go_mf'].iloc[0]
plot_go_with_relationships(selected_go_ids, godag, go2reldepth, output_file="go_by-level", dpi=300)

Plot saved as go_by-level.png with 300 DPI
